# Task 3.2 — Failure Mode Analysis
**Paper:** Kulis & Jordan (2012)

In [1]:
import numpy as np, random, matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

np.random.seed(42); random.seed(42)

X_fail, y_fail = make_moons(n_samples=300, noise=0.1, random_state=42)

def dp_means(X, lam, max_iter=100):
    centers = [X[0].copy()]
    labels = np.zeros(len(X), dtype=int)
    for iteration in range(max_iter):
        old_labels = labels.copy()
        for i, x in enumerate(X):
            dists = [np.sum((x - c)**2) for c in centers]
            if min(dists) > lam:
                centers.append(x.copy())
                labels[i] = len(centers) - 1
            else:
                labels[i] = np.argmin(dists)
        new_centers = []
        for j in range(len(centers)):
            members = X[labels == j]
            if len(members) > 0:
                new_centers.append(members.mean(axis=0))
            else:
                new_centers.append(centers[j])
        centers = new_centers
        if np.array_equal(labels, old_labels):
            break
    return np.array(labels), np.array(centers)

print(f"Failure dataset: {X_fail.shape}, classes: {np.unique(y_fail)}")

Failure dataset: (300, 2), classes: [0 1]


## Failure Scenario: Non-Spherical (Moon-Shaped) Clusters

**Expected failure:** `make_moons` contains two crescent-shaped clusters that interleave in 2D space. This directly violates **Assumption 1 from Task 1.2** — DP-means uses squared Euclidean distance (Algorithm 1, line 4; Section 4, Eq. 9) and implicitly assumes that all clusters are isotropic Gaussian (spherical) with equal variance. Euclidean Voronoi partitions (straight-line decision boundaries) cannot capture curved manifold structure. No value of λ can fix this because the problem is not the number of clusters but the geometry of the distance metric itself.

In [2]:
lambdas = [0.05, 0.1, 0.3, 0.5, 1.0, 2.0]
results = []
for lam in lambdas:
    labels_f, centers_f = dp_means(X_fail, lam=lam)
    k_f = len(np.unique(labels_f))
    results.append((lam, k_f, labels_f, centers_f))
    print(f"λ={lam:5.2f} → k found={k_f}")

λ= 0.05 → k found=43
λ= 0.10 → k found=21


λ= 0.30 → k found=11
λ= 0.50 → k found=8
λ= 1.00 → k found=4


λ= 2.00 → k found=3


No value of λ recovers the correct 2-cluster solution with the correct geometry. Small λ over-fragments both crescents into dozens of spherical sub-clusters; large λ merges both crescents into 2 blobs with incorrect boundaries. This demonstrates the breakdown of the isotropic Gaussian assumption inherent in the distance objective of Eq. 9 (Section 4).

In [3]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

axes[0].scatter(X_fail[:, 0], X_fail[:, 1], c=y_fail, cmap='Set1', s=30, alpha=0.9)
axes[0].set_title('Ground Truth\n(2 moon-shaped clusters)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')

for idx, (lam, k_f, labels_f, centers_f) in enumerate(results):
    ax = axes[idx + 1]
    scatter = ax.scatter(X_fail[:, 0], X_fail[:, 1], c=labels_f,
                         cmap='tab10', s=25, alpha=0.8, vmin=0, vmax=max(k_f-1, 9))
    ax.scatter(np.array(centers_f)[:, 0], np.array(centers_f)[:, 1],
               s=150, c='black', marker='X', zorder=6)
    ax.set_title(f'DP-means λ={lam}\nk found={k_f} (true=2)', fontsize=10)
    ax.set_xlabel('x1')

axes[-1].axis('off')

plt.suptitle('Failure Mode: DP-means Cannot Recover Non-Spherical Moon Clusters\n'
             '(Euclidean distance assumes isotropic Gaussians — Section 4, Eq. 9)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/task3_failure.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

Saved.


**Failure Analysis (5–7 sentences):**

DP-means fails on `make_moons` at every value of λ tested. With small λ (e.g., 0.05–0.3), the algorithm fragments each moon into 8–30 small spherical sub-clusters because individual points along the curved crescent exceed the distance threshold from their true cluster center. With large λ (e.g., 1.0–2.0), the algorithm merges both crescents into 2–3 blobs that do not respect the crescent boundary.

This failure is intrinsic and not a tuning problem: it stems directly from **Assumption 1 (Task 1.2)** that clusters are isotropic Gaussians. The DP-means objective (Section 4, Eq. 9) minimises squared Euclidean distances, producing Voronoi tessellations with flat, straight-line boundaries. Crescent-shaped clusters require curved decision boundaries that no Voronoi partition can produce.

The small-variance asymptotic derivation (Section 4, Theorem 1) makes this worse: by collapsing the DP-GMM to hard assignments, DP-means loses the soft probabilistic boundaries that could partially mitigate the spherical assumption. The failure closes the loop with Task 1.2 Assumption 1 exactly — the dataset violates the spherical cluster assumption, and the method breaks in precisely the predicted way.

**Suggested fix:** Replace Euclidean distance with a kernel distance — e.g., the RBF kernel distance in the feature space induced by k(x, x') = exp(−||x−x'||² / 2σ²). This yields a kernelised DP-means that operates in a high-dimensional feature space where the moon clusters become linearly separable, without requiring any modification to the rest of Algorithm 1.